# Purple-Defringe — Colab GPU runner

Runs the ONNX defringe model over a video on a Colab GPU and writes the result back to Google Drive.
Uses **batched inference + threaded decode/encode** so the GPU stays fed (a serial batch-1 loop
leaves the GPU ~15% utilised).

**Setup (do this first):**
1. `Runtime → Change runtime type → GPU` (T4 works; A100 is faster).
2. Upload `defringe.onnx` and your video to a folder on your Google Drive.
3. Edit the **Config** cell, then run all cells top-to-bottom.

### 1. Install dependencies
`ffmpeg` is preinstalled on Colab; we just need the GPU build of ONNX Runtime.

In [ ]:
!pip -q install onnxruntime-gpu
import onnxruntime as ort
print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())
!ffmpeg -version | head -n1

### 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Config — edit these

In [ ]:
# Drive lives at /content/drive/MyDrive/...
MODEL_GDRIVE  = "/content/drive/MyDrive/defringe/defringe.onnx"
VIDEO_GDRIVE  = "/content/drive/MyDrive/defringe/sanguo-ep01-10min.mp4"
OUTPUT_GDRIVE = "/content/drive/MyDrive/defringe/sanguo-defringed.mp4"

DURATION = None      # seconds from the start; None = the WHOLE video
BATCH    = 16       # frames per GPU call. Bump to 16/32 if VRAM allows (each 1080p frame ~ small)
CRF      = 16      # H.264 quality (lower = better/larger)
KEEP_AUDIO = True

### 4. Stage files on local disk
Drive's FUSE mount is slow for ffmpeg's reads, so copy locally first.

In [ ]:
import os, shutil, time
os.makedirs('/content/work', exist_ok=True)
MODEL        = '/content/work/model.onnx'
VIDEO        = '/content/work/input' + os.path.splitext(VIDEO_GDRIVE)[1]
OUTPUT_LOCAL = '/content/work/output.mp4'
print('copying from Drive to local disk...')
shutil.copy(MODEL_GDRIVE, MODEL); shutil.copy(VIDEO_GDRIVE, VIDEO)
print('model:', os.path.getsize(MODEL)//1024, 'KB |', 'video:', os.path.getsize(VIDEO)//(1024*1024), 'MB')

### 5. Create the session & confirm the GPU is active

In [ ]:
import onnxruntime as ort
PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
_sess = ort.InferenceSession(MODEL, providers=PROVIDERS)
print('using:', _sess.get_providers())
if 'CUDAExecutionProvider' not in _sess.get_providers():
    print('\u26a0\ufe0f  CUDA EP not active -> running on CPU (much slower).')
    print('   Fix: Runtime -> Change runtime type -> GPU, then re-run the install cell.')

### 6. Run the defringe pipeline (batched + threaded)
Decoder thread \u2192 batched ONNX inference \u2192 encoder thread, with a live progress bar.

In [ ]:
import subprocess, json, threading, queue, time
import numpy as np
from tqdm.auto import tqdm

def probe(path):
    out = subprocess.check_output(['ffprobe','-v','error','-select_streams','v:0',
        '-show_entries','stream=width,height,r_frame_rate,nb_frames','-of','json', path])
    s = json.loads(out)['streams'][0]
    num, den = s['r_frame_rate'].split('/')
    return int(s['width']), int(s['height']), s['r_frame_rate'], float(num)/float(den), int(s.get('nb_frames') or 0)

def defringe_video(inp, out, model, duration=None, batch=8, crf=16, audio=True):
    W, H, rate, fps, nb = probe(inp)
    total = int(round(duration*fps)) if duration else (nb or None)
    fsz = W*H*3
    print(f'{W}x{H} @ {fps:.3f} fps |', f'first {duration}s' if duration else 'full', f'| batch {batch}')

    dec = ['ffmpeg','-v','error'] + (['-t',str(duration)] if duration else []) \
        + ['-i', inp, '-f','rawvideo','-pix_fmt','rgb24','-']
    enc = ['ffmpeg','-v','error','-y','-f','rawvideo','-pix_fmt','rgb24','-s',f'{W}x{H}','-r',rate,'-i','-']
    if audio:
        enc += (['-t',str(duration)] if duration else []) + ['-i', inp, '-map','0:v:0','-map','1:a:0?','-c:a','aac','-shortest']
    enc += ['-c:v','libx264','-crf',str(crf),'-pix_fmt','yuv420p', out]

    dp = subprocess.Popen(dec, stdout=subprocess.PIPE)
    ep = subprocess.Popen(enc, stdin=subprocess.PIPE)
    sess = ort.InferenceSession(model, providers=PROVIDERS)
    iname = sess.get_inputs()[0].name
    q_in, q_out = queue.Queue(maxsize=4), queue.Queue(maxsize=4)

    def reader():
        buf_b = []
        while True:
            buf = dp.stdout.read(fsz)
            if len(buf) < fsz: break
            buf_b.append(np.frombuffer(buf, np.uint8).reshape(H, W, 3))
            if len(buf_b) == batch:
                q_in.put(np.stack(buf_b, 0)); buf_b = []
        if buf_b: q_in.put(np.stack(buf_b, 0))
        q_in.put(None)

    def writer():
        while True:
            it = q_out.get()
            if it is None: break
            ep.stdin.write(it)

    threading.Thread(target=reader, daemon=True).start()
    threading.Thread(target=writer, daemon=True).start()

    bar = tqdm(total=total, unit='frame'); n = 0; t0 = time.perf_counter()
    while True:
        b = q_in.get()
        if b is None: break
        y = sess.run(None, {iname: b})[0]   # uint8 NHWC in -> uint8 NHWC out (norm on-device)
        q_out.put(y.tobytes()); n += len(b); bar.update(len(b))
        bar.set_postfix_str(f'{(time.perf_counter()-t0)/n*1000:.0f} ms/frame')
    q_out.put(None); bar.close()
    ep.stdin.close(); dp.stdout.close(); ep.wait(); dp.wait()
    print(f'done: {n} frames in {time.perf_counter()-t0:.0f}s')
    return out

defringe_video(VIDEO, OUTPUT_LOCAL, MODEL, duration=DURATION, batch=BATCH, crf=CRF, audio=KEEP_AUDIO)

### 7. Save the result back to Google Drive

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_GDRIVE), exist_ok=True)
shutil.copy(OUTPUT_LOCAL, OUTPUT_GDRIVE)
print('saved ->', OUTPUT_GDRIVE, '(', os.path.getsize(OUTPUT_GDRIVE)//(1024*1024), 'MB )')